In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(
    "GSAI-ML/LLaDA-8B-Base", trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    "GSAI-ML/LLaDA-8B-Base",
    trust_remote_code=True,
    torch_dtype=torch.float32,  # for MPS
    device_map="mps",
)


config.json: 0.00B [00:00, ?B/s]

configuration_llada.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Base:
- configuration_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


modeling_llada.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Base:
- modeling_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)

model-00006-of-00006.safetensors:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

model-00005-of-00006.safetensors:   0%|          | 0.00/2.92G [00:00<?, ?B/s]

  2025-12-05T18:43:22.070976Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/reconstructions/6bfacbf69a6f931caa43e0120cfb409b9946a2fbf8bd180cfc42eaca3df05e4d", source: hyper_util::client::legacy::Error(Connect, ConnectError("dns error", Custom { kind: Uncategorized, error: "failed to lookup address information: nodename nor servname provided, or not known" })) }). Retrying...
    at /Users/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:233

  2025-12-05T18:43:22.071050Z  WARN  Retry attempt #0. Sleeping 2.842284979s before the next attempt
    at /Users/runner/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171

  2025-12-05T18:43:22.070968Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://cas-server.xethub.hf.co/reconstructions/c445f45c0af81a528facfa1031103c6223937605d12449e8c417153940967496", source: hyper_util::client::legacy::Error(Connect, ConnectError("dns error", Custom { ki

RuntimeError: Invalid buffer size: 29.86 GB

In [3]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

model_id = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_id)
mlm_model = AutoModelForMaskedLM.from_pretrained(model_id)

device = "mps" if torch.backends.mps.is_available() else "cpu"
mlm_model.to(device)
mlm_model.eval()


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [4]:
# masked_lm_core.py

import numpy as np
import torch
import torch.nn.functional as F

from diffusion_core import Core, ModelConfig


@torch.no_grad()
def masked_lm_single_step(
    model,
    input_ids: torch.Tensor,
    *,
    mask_token_id: int,
    pad_token_id: int | None,
    mask_frac: float,
) -> torch.Tensor:
    """
    One 'diffusion' step with a masked LM:
    - pick a random subset of non-pad positions
    - mask them
    - run MLM
    - sample new tokens for those positions

    Args:
        model: AutoModelForMaskedLM
        input_ids: (B, L) int64
        mask_token_id: tokenizer.mask_token_id
        pad_token_id: tokenizer.pad_token_id or None
        mask_frac: fraction (0,1] of eligible tokens to mask per sequence

    Returns:
        next_ids: (B, L) int64 updated sequence
    """
    device = next(model.parameters()).device
    input_ids = input_ids.to(device)
    B, L = input_ids.shape

    # eligible positions = not pad
    if pad_token_id is None:
        eligible = torch.ones_like(input_ids, dtype=torch.bool, device=device)
    else:
        eligible = input_ids != pad_token_id

    # build mask_positions per row
    mask_positions = torch.zeros_like(input_ids, dtype=torch.bool, device=device)

    for b in range(B):
        idx = torch.nonzero(eligible[b], as_tuple=False).view(-1)
        if idx.numel() == 0:
            continue
        k = max(1, int(mask_frac * idx.numel()))
        perm = torch.randperm(idx.numel(), device=device)
        chosen = idx[perm[:k]]
        mask_positions[b, chosen] = True

    masked_input = input_ids.clone()
    masked_input[mask_positions] = mask_token_id

    outputs = model(masked_input)
    logits = outputs.logits  # (B, L, V)
    probs = F.softmax(logits, dim=-1)

    # sample new tokens only at masked positions
    # flatten masked positions for vectorized sampling
    flat_mask = mask_positions.view(-1)
    if flat_mask.sum() == 0:
        return input_ids  # nothing to update

    logits_flat = logits.view(-1, logits.size(-1))[flat_mask]  # (M, V)
    probs_flat = F.softmax(logits_flat, dim=-1)
    sampled_flat = torch.multinomial(probs_flat, num_samples=1).squeeze(-1)  # (M,)

    next_ids = input_ids.clone().view(-1)
    next_ids[flat_mask] = sampled_flat
    next_ids = next_ids.view(B, L)

    return next_ids


In [ ]:
import torch
import torch.nn.functional as F

device = next(mlm_model.parameters()).device

text = f"An interesting arithmetic fact is that {tokenizer.mask_token} {tokenizer.mask_token} {tokenizer.mask_token} {tokenizer.mask_token} {tokenizer.mask_token}"
input_ids = tokenizer.encode(text, add_special_tokens=True, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = mlm_model(input_ids)
    logits = outputs.logits  # (1, L, V)

mask_positions = (input_ids == tokenizer.mask_token_id)  # (1, L)
probs = F.softmax(logits, dim=-1)

# sample new token for the [MASK] position(s)
sampled_ids = input_ids.clone()
for (b, l) in zip(*mask_positions.nonzero(as_tuple=True)):
    token_probs = probs[b, l]
    sampled_token = torch.multinomial(token_probs, 1)
    sampled_ids[b, l] = sampled_token

print("Before (raw):", tokenizer.decode(input_ids[0],  skip_special_tokens=False))
print("After  (raw):", tokenizer.decode(sampled_ids[0], skip_special_tokens=False))
print("Before:", tokenizer.decode(input_ids[0],  skip_special_tokens=True))
print("After :", tokenizer.decode(sampled_ids[0], skip_special_tokens=True))


Before (raw): [CLS] one plus [MASK] is twelve [SEP]
After  (raw): [CLS] one plus one is twelve [SEP]
Before: one plus is twelve
After : one plus one is twelve


In [5]:
class MaskedLMCore(Core):
    """
    A diffusion-like core using a masked language model (BERT, DeBERTa, etc.).

    State: token IDs of shape (S, L), stored as np.int64.
    At each diffusion step t, we:
      - mask a fraction of tokens
      - sample them from the MLM
    """

    def __init__(
        self,
        config: ModelConfig,
        tokenizer,
        mlm_model,
        S: int = 32,
        N: int = 4,
        load_from_config: bool = False,
    ):
        super().__init__(config)
        self.tokenizer = tokenizer
        self.mlm_model = mlm_model

        if load_from_config:
            # config is a plain dict when loaded from JSON
            self._from_config_dict(config)
        else:
            # config is a ModelConfig when constructed manually
            self.model_config = config.to_dict()
            self.state_shape = tuple(config.state_shape)
            self.index_set = config.index_set
            self.deets = config.deets

        self.S = S
        self.N = N
        self.T = self.deets.get("T", 20)
        self.max_length = self.state_shape[0]

        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id

        # optional: treat first prompt_len tokens as fixed (never masked)
        self.prompt_len = self.deets.get("prompt_len", 0)

        self.X = None          # (S, L) np.int64
        self.tentative_states = None
        self.t = None

    # ---------- internal helpers ----------

    def _from_config_dict(self, cfg_dict: dict):
        """Called when reconstructing core from saved JSON config."""
        self.model_config = cfg_dict
        self.state_shape = tuple(cfg_dict["state_shape"])
        self.index_set = cfg_dict["index_set"]
        self.deets = cfg_dict.get("deets", {})
        self.T = self.deets.get("T", 20)
        self.max_length = self.state_shape[0]

    def _mask_schedule(self, t: int) -> float:
        """
        Simple linear schedule: high masking early, low masking near t=0.
        t goes from T down to 0 (like your MixtureModel).
        """
        max_mask = self.deets.get("max_mask_frac", 0.8)
        min_mask = self.deets.get("min_mask_frac", 0.1)
        return min_mask + (max_mask - min_mask) * (t / self.T)

    # ---------- Core interface ----------

    def load_from_config(self, config_path):
        with open(config_path, "r", encoding="utf-8") as f:
            cfg_dict = json.load(f)
        self._from_config_dict(cfg_dict)

    def reset(self):
        """
        Start from all [MASK] tokens (plus optional prompt).
        """
        self.t = self.T
        # Initialize X as all pads, then fill any prompt externally if you want.
        self.X = np.full((self.S, self.max_length),
                         self.mask_token_id,
                         dtype=np.int64)
        self.tentative_states = None

    def propose(self, S: int = 1):
        """
        Generate N proposals from the current X_t.
        Return shape (S, N, L) as in MixtureModel.
        """
        if self.t < 0:
            raise ValueError("No remaining timesteps to propose.")
        if self.X is None:
            # If user forgot to call reset, do it now.
            self.reset()

        t = self.t
        mask_frac = self._mask_schedule(t)

        # current state as torch
        device = next(self.mlm_model.parameters()).device
        x_torch = torch.from_numpy(self.X).long().to(device)

        proposals = []
        for _ in range(self.N):
            x_next = masked_lm_single_step(
                self.mlm_model,
                x_torch,
                mask_token_id=self.mask_token_id,
                pad_token_id=self.pad_token_id,
                mask_frac=mask_frac,
            )
            proposals.append(x_next.cpu())

        # (N, S, L) -> (S, N, L)
        proposals = torch.stack(proposals, dim=0).permute(1, 0, 2).numpy()
        self.tentative_states = proposals
        return proposals

    def propose_external(self, init_state, time, N: int = 1):
        """
        Analogous to MixtureModel.propose_external:
        given init_state at time 'time', generate N proposals.
        init_state: (S, L) np.int64
        time: integer index t
        Returns: (S, L) if N == 1 else (S, N, L)
        """
        X = np.asarray(init_state, dtype=np.int64)
        mask_frac = self._mask_schedule(time)

        device = next(self.mlm_model.parameters()).device
        x_torch = torch.from_numpy(X).long().to(device)

        if N == 1:
            x_next = masked_lm_single_step(
                self.mlm_model,
                x_torch,
                mask_token_id=self.mask_token_id,
                pad_token_id=self.pad_token_id,
                mask_frac=mask_frac,
            )
            return x_next.cpu().numpy()
        else:
            proposals = []
            for _ in range(N):
                x_next = masked_lm_single_step(
                    self.mlm_model,
                    x_torch,
                    mask_token_id=self.mask_token_id,
                    pad_token_id=self.pad_token_id,
                    mask_frac=mask_frac,
                )
                proposals.append(x_next.cpu())
            proposals = torch.stack(proposals, dim=0).permute(1, 0, 2).numpy()
            return proposals

    def commit(self, X):
        """
        Accept a chosen proposal X of shape (S, L) and move to t-1.
        """
        X = np.asarray(X, dtype=np.int64)
        if X.shape != (self.S, self.max_length):
            raise ValueError(f"commit expected shape {(self.S, self.max_length)}, got {X.shape}")
        self.X = X
        self.t -= 1

    def collect_run(self):
        """
        Unguided diffusion run, analogous to MixtureModel.collect_run().
        Returns states of shape (T+1, S, L), with index 0 corresponding to t=0.
        """
        states = []
        self.reset()
        while self.t >= 0:
            proposals = self.propose()
            # take first proposal as unguided trajectory
            self.commit(proposals[:, 0, :])
            states.append(self.X.copy())
        self.last_run = states
        # flip so that states[0] is t=0, like mixture_model.py
        return np.flip(np.array(states), axis=0)

    def save(self, states, data_dir="data", run_id=None) -> dict:
        """
        Same pattern as MixtureModel.save: save config JSON and states .npy
        """
        os.makedirs(data_dir, exist_ok=True)
        if run_id is None:
            run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        cfg_path = os.path.join(data_dir, f"{run_id}_config.json")
        states_path = os.path.join(data_dir, f"{run_id}_states.npy")

        def _to_serializable(obj):
            if hasattr(obj, "to_dict"):
                return _to_serializable(obj.to_dict())
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            if isinstance(obj, (np.floating, np.integer)):
                return obj.item()
            if isinstance(obj, dict):
                return {k: _to_serializable(v) for k, v in obj.items()}
            if isinstance(obj, (list, tuple)):
                return [_to_serializable(v) for v in obj]
            return obj

        with open(cfg_path, "w", encoding="utf-8") as f:
            json.dump(_to_serializable(self.model_config), f, indent=2)

        np.save(states_path, np.asarray(states))
        return {"id": run_id, "config": cfg_path, "states": states_path}


In [14]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

from diffusion_core import ModelConfig

# 1. Device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

# 2. Load tokenizer + masked LM (BERT)
hf_model_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(hf_model_id)
mlm_model = AutoModelForMaskedLM.from_pretrained(hf_model_id)
mlm_model.to(device)
mlm_model.eval()

# 3. Build a ModelConfig for the masked LM diffusion core
T = 5                 # number of diffusion steps
max_length = 16       # sequence length
S = 2                 # batch size
N = 3                 # proposals per step

cfg = ModelConfig()
cfg.model_name = "masked_lm"
cfg.state_shape = (max_length,)
cfg.index_set = list(range(T + 1))
cfg.deets = {
    "T": T,
    "hf_model_id": hf_model_id,
    "max_mask_frac": 0.8,
    "min_mask_frac": 0.2,
    "prompt_len": 0,
}

# 4. Instantiate the core
core = MaskedLMCore(cfg, tokenizer, mlm_model, S=S, N=N)

# 5. Reset to initial all-[MASK] state
core.reset()

# 6. Run ONE propose step and inspect the first proposal of the first sequence
proposals = core.propose()          # shape: (S, N, L)
sample_ids = proposals[0, 0]        # (L,)

print("Token IDs:", sample_ids.tolist())
print("Decoded text:", tokenizer.decode(sample_ids, skip_special_tokens=True))

# 7. (Optional) run a full unguided trajectory and inspect final sample
states = core.collect_run()         # shape: (T+1, S, L), index 0 = t=0
final_ids = states[0, 0]            # final sample for seq 0

print("Final sample IDs:", final_ids.tolist())
print("Final sample text:", tokenizer.decode(final_ids, skip_special_tokens=True))


Using device: mps


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Token IDs: [103, 1996, 28301, 3371, 6642, 1000, 103, 1007, 4756, 103, 103, 9411, 1010, 1029, 2116, 5538]
Decoded text: the quakers minute occasions " ) laugh dealt,? many borough
Final sample IDs: [2480, 14998, 18395, 22044, 2162, 6191, 29348, 2072, 1007, 103, 9413, 10556, 9152, 2340, 8663, 2852]
Final sample text: ##z butcher tattooshorpe war 63 irritatingi ) er ka ni 11con dr
